# File and Data Field Descriptions

- **train.csv** - Personal records for about two-thirds (~8700) of the passengers, to be used as training data.
  - **PassengerId** - A unique Id for each passenger. Each Id takes the form gggg\*pp where \*\*\_gggg indicates a group the passenger is travelling**\* with and **_pp is their number within the group_\*\*. People in a group are often family members, but not always.
  - **HomePlanet** - The planet the passenger departed from, typically their planet of permanent residence.
  - **CryoSleep** - Indicates whether the passenger elected to be put into suspended animation for the duration of the voyage. Passengers in cryosleep are confined to their cabins.
  - **Cabin** - The cabin number where the passenger is staying. Takes the form **_deck/num/side, where side can be either P for Port or S for Starboard_**.
  - **Destination** - The planet the passenger will be debarking to.
  - **Age** - The age of the passenger.
  - **VIP** - Whether the passenger has **_paid for special VIP service_** during the voyage.
  - **RoomService, FoodCourt, ShoppingMall, Spa, VRDeck** - Amount the passenger has billed at each of the Spaceship Titanic's many luxury amenities.
  - **Name** - The first and last names of the passenger.
  - **Transported** - Whether the passenger was transported to another dimension. **This is the target**, the column you are trying to predict.
- **test.csv** - Personal records for the remaining one-third (~4300) of the passengers, to be used as test data. Your task is to predict the value of Transported for the passengers in this set.
- **sample_submission.csv** - A submission file in the correct format.
  - **PassengerId** - Id for each passenger in the test set.
  - **Transported** - The target. For each passenger, predict either True or False.


**Todo:**

- Unbekannte Werte auffuellen
  - Name: werte koennen mit der PassengerId ergaenzt werden
  - HomePlanet: kann man dem haeufigsten vorkommenden ergaenzt werden "Eearth"
  - CryoSleep: Wenn Nachname gleich oder Cabin gleich dann Mehrheitsbestimmung, sonst "False" setzten.
- Kombiniere "RoomService, FoodCourt, ShoppingMall, Spa, VRDeck" zu Spalte "Total"
- one-hot:
  - VIP, da nur T/F
  - CryoSleep
  - HomePlanet -> droppen, da noise
  - Destination -> droppen, da noise
- Age zu bins
- Cabins:
  - Bei Duplikaten zu Spalte "FamilySize" hinzufügen
  - Deck koennte man binen
  - side zu one-hot
  - num koennte man droppen oder zu bins umwandeln
- Name: eventuell zu "isAlone" umwandeln und droppen
- PassengerId: unterteilen in Gruppe wie "\_gggg"


In [2]:
# data analysis and wrangling
import pandas as pd
import numpy as np
import random as rnd

# visualization
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

# machine learning
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import Perceptron, SGDClassifier, LogisticRegression
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix


SEED = 42
np.random.seed(SEED)

In [68]:
df_train = pd.read_csv("./data/spaceship-titanic/train.csv")
df_test = pd.read_csv("./data/spaceship-titanic/test.csv")

labels = df_train["Transported"]
# y_test = df_test["Transported"]

# df_train.drop(["Transported"], axis=1, inplace=True)
# df_test.drop(["Transported"], axis=1, inplace=True)

In [64]:
labels

0       False
1        True
2       False
3       False
4        True
        ...  
8688    False
8689    False
8690     True
8691    False
8692     True
Name: Transported, Length: 8693, dtype: bool

In [69]:
combined = pd.concat([df_train, df_test], sort=False).reset_index(drop=True)
combined.isnull().sum()

PassengerId        0
HomePlanet       288
CryoSleep        310
Cabin            299
Destination      274
Age              270
VIP              296
RoomService      263
FoodCourt        289
ShoppingMall     306
Spa              284
VRDeck           268
Name             294
Transported     4277
dtype: int64

In [ ]:
# Jeder Passanger ist einzigartig.
#
combined.select_dtypes(include=["O"]).nunique()

PassengerId    12970
HomePlanet         3
CryoSleep          2
Cabin           9825
Destination        3
VIP                2
Name           12629
dtype: int64

In [24]:
combined.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,12700.000000,12707.000000,12681.000000,12664.000000,12686.000000,12702.000000
mean,28.771969,222.897852,451.961675,174.906033,308.476904,306.789482
std,14.387261,647.596664,1584.370747,590.558690,1130.279641,1180.097223
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,49.000000,77.000000,29.000000,57.000000,42.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [20]:
labels.describe()

count     8693
unique       2
top       True
freq      4378
Name: Transported, dtype: object

### Looking for ways to fill missing values


In [46]:
pd.concat(g for _, g in combined.groupby("Name") if len(g) > 1)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
2404,2589_01,Europa,True,C/91/P,TRAPPIST-1e,26.0,False,0.0,0.0,0.0,0.0,0.0,Alasmon Meteet
9286,1219_02,Europa,True,B/52/S,55 Cancri e,23.0,False,0.0,0.0,0.0,0.0,0.0,Alasmon Meteet
6296,6665_01,Europa,True,B/222/P,55 Cancri e,17.0,False,0.0,0.0,0.0,0.0,0.0,Alraium Disivering
7270,7775_01,Europa,False,C/253/P,55 Cancri e,28.0,False,7.0,489.0,0.0,4.0,6027.0,Alraium Disivering
476,0512_02,Europa,True,D/18/S,TRAPPIST-1e,55.0,False,0.0,0.0,0.0,0.0,0.0,Ankalik Nateansive
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3381,3640_02,Earth,True,G/589/S,TRAPPIST-1e,41.0,False,0.0,0.0,0.0,0.0,0.0,Troya Schwardson
895,0964_02,Europa,True,B/36/P,TRAPPIST-1e,48.0,False,0.0,0.0,0.0,0.0,0.0,Weidus Platch
10002,2808_01,Europa,True,C/95/P,55 Cancri e,30.0,False,0.0,0.0,0.0,0.0,0.0,Weidus Platch
6231,6591_01,Earth,False,F/1261/S,TRAPPIST-1e,47.0,False,285.0,600.0,3.0,0.0,640.0,Willy Mcfarleys


In [58]:
cabins = combined.copy()
# cabins["Deck"] = cabins["Cabin"].apply(lambda x: str(x).split("/")[0] if pd.notnull(x) else x)

cabins["FamilySize"] = cabins.groupby("Cabin")["Cabin"].transform("count")

In [63]:
cabins[cabins["FamilySize"] > 1].sort_values("Cabin", ascending=False)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,FamilySize
11904,7046_01,Europa,False,T/3/S,55 Cancri e,27.0,NaN,6.0,400.0,0.0,6472.0,0.0,Tope Ativeezy,3.0
11906,7046_03,Europa,False,T/3/S,TRAPPIST-1e,47.0,False,0.0,339.0,0.0,508.0,2000.0,Zinon Ativeezy,3.0
11905,7046_02,Europa,False,T/3/S,55 Cancri e,44.0,False,0.0,1190.0,0.0,1906.0,167.0,Genubih Ativeezy,3.0
5826,6168_02,Earth,False,G/999/S,55 Cancri e,12.0,False,0.0,0.0,0.0,0.0,0.0,Glenna Gordond,2.0
5825,6168_01,Earth,True,G/999/S,PSO J318.5-22,16.0,False,0.0,0.0,0.0,0.0,0.0,Karie Gordond,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52,0056_01,Europa,False,A/1/S,TRAPPIST-1e,2.0,False,0.0,0.0,0.0,0.0,0.0,Okulas Tractive,3.0
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,2.0
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,2.0
114,0119_02,Europa,True,A/0/P,TRAPPIST-1e,13.0,False,0.0,0.0,0.0,0.0,0.0,Kleeiak Coning,2.0


In [71]:
combined[["CryoSleep", "Transported"]].groupby(["CryoSleep"], as_index=False).mean().sort_values(
    by="Transported", ascending=False
)

,CryoSleep,Transported
1,True,0.817583
0,False,0.328921


In [ ]:
combined[["CryoSleep", "Transported"]].groupby(["CryoSleep"], as_index=False).mean().sort_values(
    by="Transported", ascending=False
)

In [72]:
combined

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12965,9266_02,Earth,True,G/1496/S,TRAPPIST-1e,34.0,False,0.0,0.0,0.0,0.0,0.0,Jeron Peter,NaN
12966,9269_01,Earth,False,NaN,TRAPPIST-1e,42.0,False,0.0,847.0,17.0,10.0,144.0,Matty Scheron,NaN
12967,9271_01,Mars,True,D/296/P,55 Cancri e,NaN,False,0.0,0.0,0.0,0.0,0.0,Jayrin Pore,NaN
12968,9273_01,Europa,False,D/297/P,NaN,NaN,False,0.0,2680.0,0.0,0.0,523.0,Kitakan Conale,NaN
